<a href="https://colab.research.google.com/github/abd500253-coder/Machine_Learning_Journey/blob/main/03.%20Model%20Evaluation%20%26%20Performance%20Metrics/Logistic%20Regression/Logistic%20Regression%20vs%20Custom%20build%20Model/Logistic_Regression_vs_Custom_Build_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1. Setup and Data Generation**
First, we import the necessary libraries and generate a synthetic classification dataset. We add a column of ones to our feature matrix X to account for the intercept (bias) term during our matrix multiplication later.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification

# Generate synthetic binary classification data
X, y = make_classification(
    n_samples=2000,
    n_features=8,
    n_informative=5,        # 5 metrics strongly predict churn
    n_redundant=2,          # 2 metrics are highly correlated/overlapping
    n_classes=2,            # Binary choice: Churn or Stay
    class_sep=1.2,          # Clean separation (easier to predict)
    random_state=42
)

# Add an intercept column (a column of 1s) to the beginning of matrix X
X = np.insert(X, 0, 1, axis=1)

# **2. Helper Functions: Sigmoid and Log Loss**
We define the sigmoid function to map predictions to probabilities between 0 and 1. We also define the log loss (binary cross-entropy) function to evaluate the performance of our model. We include a small epsilon value to prevent mathematical errors like calculating the log of absolute zero.

In [21]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def log_loss(y_true, y_pred):
    # Epsilon is a tiny number added to prevent np.log(0) which crashes Python
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)

    # The actual log loss formula
    loss = -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    return loss

# **3. Custom Gradient Descent Implementation**
This function trains the logistic regression model from scratch. It initializes weights to zero and iteratively updates them by calculating the gradient of the loss function with respect to the weights.

In [22]:
def gradient_descent_logistic(X, y, epochs, learning_rate=0.5):
    m = X.shape[0]
    loss_history = []

    # Initialize weights to zeros (one for each feature + intercept)
    weights = np.zeros(X.shape[1])

    for i in range(epochs):
        # 1. Calculate predictions
        y_pred = sigmoid(np.dot(X, weights))

        # 2. Calculate and store the loss
        loss = log_loss(y, y_pred)
        loss_history.append(loss)

        # 3. Calculate the gradient (derivative of the loss function)
        derivative = (1/m) * np.dot(X.T, (y_pred - y))

        # 4. Update weights
        weights = weights - learning_rate * derivative

    return weights, loss_history

# **4. Model Evaluation & Scikit-Learn Comparison**
To verify that our custom gradient descent works correctly, we compare its final weights against those calculated by the industry-standard scikit-learn library.

In [23]:
from sklearn.linear_model import LogisticRegression

# 1. Run our custom implementation
custom_weights, loss_history = gradient_descent_logistic(X, y, epochs=500, learning_rate=0.1)

# 2. Run scikit-learn's Logistic Regression
# We set fit_intercept=False because our X matrix already has the column of 1s built-in.
sk_model = LogisticRegression(penalty=None, fit_intercept=False, max_iter=500)
sk_model.fit(X, y)
sk_weights = sk_model.coef_[0]

# 3. Print the weights side-by-side to check similarity
comparison_df = pd.DataFrame({
    'Feature': [f'w_{i}' for i in range(len(custom_weights))],
    'Your Model': custom_weights,
    'Scikit-Learn': sk_weights,
    'Difference': np.abs(custom_weights - sk_weights)
})

print(comparison_df)

  Feature  Your Model  Scikit-Learn  Difference
0     w_0    0.585019      0.613141    0.028123
1     w_1    0.048840      0.052671    0.003831
2     w_2    0.479199      0.474581    0.004617
3     w_3    0.023278      0.025011    0.001733
4     w_4   -0.795085     -0.804891    0.009806
5     w_5   -0.326962     -0.331721    0.004759
6     w_6   -0.177368     -0.175126    0.002242
7     w_7   -0.299847     -0.297875    0.001972
8     w_8    0.808667      0.817590    0.008923
